# Importación de librerías y carga de datos base

In [10]:

import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

df_info = pd.read_csv('./../../dataset/oulad/studentInfo.csv')
df_vle = pd.read_csv('./../../dataset/oulad/studentVle.csv')

print("Datos cargados correctamente.")

Datos cargados correctamente.


# Definición de la variable objetivo y partición cronológica

In [11]:
df_info['target'] = df_info['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)

presentaciones_2013 = ['2013B', '2013J']
presentaciones_2014 = ['2014B', '2014J']

df_train_info = df_info[df_info['code_presentation'].isin(presentaciones_2013)].copy()
df_test_info = df_info[df_info['code_presentation'].isin(presentaciones_2014)].copy()

print(f"Estudiantes históricos (Train 2013): {len(df_train_info)}")
print(f"Estudiantes nuevos (Test 2014): {len(df_test_info)}")

Estudiantes históricos (Train 2013): 13529
Estudiantes nuevos (Test 2014): 19064


# Función de extracción de características temporales

In [12]:
def preparar_datos_temporales(df_vle, df_info_split, max_dias):
    vle_filtrado = df_vle[df_vle['date'] <= max_dias]

    vle_agg = vle_filtrado.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
    vle_agg.rename(columns={'sum_click': 'total_clicks'}, inplace=True)

    df_merged = pd.merge(df_info_split, vle_agg, on=['id_student', 'code_module', 'code_presentation'], how='left')

    df_merged['total_clicks'] = df_merged['total_clicks'].fillna(0)

    X = df_merged[['total_clicks']]
    y = df_merged['target']

    return X, y

print("Función de preparación de datos lista.")

Función de preparación de datos lista.


# Ejecución del experimento completo

In [16]:
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        proba = self.estimator.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

scoring_metrics = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1-Score': make_scorer(f1_score, zero_division=0)
}

param_grid_knn = {
    'clasificador__estimator__n_neighbors': [3, 5, 9, 15],
    'clasificador__estimator__weights': ['uniform', 'distance'],
    'clasificador__threshold': [0.3, 0.4, 0.5, 0.6]
}

ventanas_temporales = [30, 60, 90]
estrategias_balanceo = ['Línea Base (Sin balanceo)', 'Undersampling', 'ADASYN']

resultados_mejores = []
todas_las_permutaciones = []

print("Iniciando búsqueda exhaustiva CP-03 (KNN)...")

for dias in ventanas_temporales:
    print(f"--- Evaluando ventana a {dias} días ---")

    X_train, y_train = preparar_datos_temporales(df_vle, df_train_info, dias)
    X_test, y_test = preparar_datos_temporales(df_vle, df_test_info, dias)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for estrategia in estrategias_balanceo:
        if estrategia == 'Undersampling':
            sampler = RandomUnderSampler(random_state=42)
        elif estrategia == 'ADASYN':
            sampler = ADASYN(random_state=42, n_neighbors=5)
        else:
            sampler = None

        pasos = []
        if sampler is not None:
            pasos.append(('sampler', sampler))

        knn_base = KNeighborsClassifier()
        wrapper = ThresholdWrapper(estimator=knn_base)
        pasos.append(('clasificador', wrapper))

        pipeline = ImbPipeline(pasos)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid_knn,
            cv=5,
            scoring=scoring_metrics,
            refit='Recall',
            n_jobs=-1,
            return_train_score=False
        )

        grid_search.fit(X_train_scaled, y_train)

        mejor_modelo = grid_search.best_estimator_
        y_pred = mejor_modelo.predict(X_test_scaled)

        resultados_mejores.append({
            'Días': dias,
            'Estrategia': estrategia,
            'Mejor K': grid_search.best_params_['clasificador__estimator__n_neighbors'],
            'Mejor Weight': grid_search.best_params_['clasificador__estimator__weights'],
            'Mejor Threshold': grid_search.best_params_['clasificador__threshold'],
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0)
        })

        cv_results = grid_search.cv_results_
        for i in range(len(cv_results['params'])):
            todas_las_permutaciones.append({
                'Días': dias,
                'Estrategia': estrategia,
                'K': cv_results['param_clasificador__estimator__n_neighbors'][i],
                'Weights': cv_results['param_clasificador__estimator__weights'][i],
                'Threshold': cv_results['param_clasificador__threshold'][i],
                'CV_Accuracy': cv_results['mean_test_Accuracy'][i],
                'CV_Precision': cv_results['mean_test_Precision'][i],
                'CV_Recall': cv_results['mean_test_Recall'][i],
                'CV_F1': cv_results['mean_test_F1-Score'][i]
            })

        print(f"  > {estrategia}: Procesado.")

df_mejores = pd.DataFrame(resultados_mejores)
df_todas_permutaciones = pd.DataFrame(todas_las_permutaciones)

mejor_fila = df_mejores.sort_values(by=['Recall', 'F1-Score'], ascending=[False, False]).iloc[0]
mejor_dia = mejor_fila['Días']
mejor_estrategia = mejor_fila['Estrategia']
mejor_k = mejor_fila['Mejor K'] # Extraemos la K ganadora

df_tradeoff = df_todas_permutaciones[
    (df_todas_permutaciones['Días'] == mejor_dia) &
    (df_todas_permutaciones['Estrategia'] == mejor_estrategia) &
    (df_todas_permutaciones['K'] == mejor_k)
].copy()

df_tradeoff = df_tradeoff.sort_values(by=['Weights', 'Threshold'])

print("\n=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===")
display(df_mejores)

print(f"\n=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO ({mejor_dia} Días | {mejor_estrategia} | K={mejor_k}) ===")
display(df_tradeoff)

Iniciando búsqueda exhaustiva CP-03 (KNN)...
--- Evaluando ventana a 30 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 60 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 90 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.

=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===


,Días,Estrategia,Mejor K,Mejor Weight,Mejor Threshold,Accuracy,Precision,Recall,F1-Score
0,30,Línea Base (Sin balanceo),3,uniform,0.3,0.570132,0.417713,0.692188,0.521012
1,30,Undersampling,15,uniform,0.3,0.383340,0.277996,0.517006,0.361573
2,30,ADASYN,15,uniform,0.3,0.556704,0.400179,0.626339,0.488345
3,60,Línea Base (Sin balanceo),3,uniform,0.3,0.576322,0.421866,0.686753,0.522664
4,60,Undersampling,15,uniform,0.3,0.416859,0.300120,0.545426,0.387189
5,60,ADASYN,15,uniform,0.3,0.501154,0.381162,0.764870,0.508781
6,90,Línea Base (Sin balanceo),3,uniform,0.3,0.597146,0.437707,0.677124,0.531707
7,90,Undersampling,15,uniform,0.3,0.423416,0.309290,0.573381,0.401828
8,90,ADASYN,15,uniform,0.3,0.521244,0.396072,0.795465,0.528832



=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO (90 Días | ADASYN | K=15) ===


,Días,Estrategia,K,Weights,Threshold,CV_Accuracy,CV_Precision,CV_Recall,CV_F1
284,90,ADASYN,15,distance,0.3,0.610684,0.385410,0.671896,0.484895
285,90,ADASYN,15,distance,0.4,0.663607,0.426718,0.594195,0.489509
286,90,ADASYN,15,distance,0.5,0.704263,0.467491,0.493508,0.473346
287,90,ADASYN,15,distance,0.6,0.741813,0.545120,0.407130,0.459052
280,90,ADASYN,15,uniform,0.3,0.472022,0.318412,0.808454,0.455088
281,90,ADASYN,15,uniform,0.4,0.535736,0.340989,0.731545,0.462421
282,90,ADASYN,15,uniform,0.5,0.655702,0.412753,0.562276,0.469929
283,90,ADASYN,15,uniform,0.6,0.699978,0.462614,0.469438,0.458302
